# **Request Pool**

Builds the pool of 5,042 candidate requests from seven public sources (RouterArena, Magpie-Llama-3.1-Pro-300K-Filtered, MATH, TACO, Dolly-15k, LongBench v2, LongBench v1) and saves it as one Parquet file. The two LongBench sources stay in the pool but are excluded from the experiments, which use 4,319 requests.

Requests whose source has no domain keep `domain = None` and are treated as general-purpose by the routing pipeline, which also maps the difficulty labels to the complexity $h_i$.

## Install

In [4]:
!pip install -q "datasets>=3.0" transformers pyarrow huggingface_hub

## Imports and configuration

In [ ]:
import ast
import json
import os
import re
import unicodedata
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

SEED = 42
CELL_SIZE = 50                 # requests per (source, task, domain, difficulty) cell; all of them if the cell has fewer
MAX_INPUT_TOKENS = 128_000     # largest context window of the model pool
CHARS_PER_TOKEN_UPPER = 6      # loose bound for a cheap pre-filter: longer texts are certainly over the token limit
TOKENIZER_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated: accept the Meta license on Hugging Face once
OUTPUT_DIR = "data/request_pool"
POOL_VERSION = "v2"

SOURCES = {
    "routerarena":  {"repo": "RouteWorks/RouterArena", "split": "full"},
    "magpie":       {"repo": "Magpie-Align/Magpie-Llama-3.1-Pro-300K-Filtered", "split": "train"},
    "math":         {"repo": "EleutherAI/hendrycks_math", "split": ["train", "test"]},
    "taco":         {"repo": "XiaoBanni/TACO_with_solution", "split": "train"},
    "dolly":        {"repo": "databricks/databricks-dolly-15k", "split": "train"},
    "longbench_v2": {"repo": "zai-org/LongBench-v2", "split": "train"},
    "longbench_v1": {"repo": "zai-org/LongBench", "file": "data.zip"},
}

# Commits of the source datasets used in the thesis (None loads the latest version).
REVISIONS = {
    "routerarena": "a4a062ce3313b56bb09c042e1bc37b61d34e3bd8",
    "magpie": "1a982eea9ece373700dd8dfd04a4de08c2578c24",
    "math": "21a5633873b6a120296cce3e2df9d5550074f4a3",
    "taco": "c1d27565ce784104500854907dd35048c8d86b21",
    "dolly": "bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a",
    "longbench_v2": "2b48e494f2c7a2f0af81aae178e05c7e1dde0fe9",
    "longbench_v1": "5e628be450b7e67fb7ae6e201bd6d8f7056f7672",
}

BUILD_LOG = {}   # per-source counts at each step, saved with the pool

Notes on the source choices:
- MATH: the original `hendrycks/competition_math` was removed from Hugging Face; the EleutherAI copy has the same problems, split by subject.
- TACO: the original `BAAI/TACO` uses a custom loading script that recent versions of `datasets` no longer run. `XiaoBanni/TACO_with_solution` is a Parquet copy with the same columns, holding 9,095 of the 26,443 problems.
- LongBench v1: also distributed with a loading script, so the notebook downloads the original `data.zip` from the same repository and reads the JSON Lines files directly. Only the LongBench-E files (`*_e.jsonl`) are used, because they are sampled evenly over three length ranges.

## Hugging Face token

In [6]:
try:
    from google.colab import userdata                 # Colab: secret HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:                                    # local run: environment variable or prompt
    from getpass import getpass
    os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output folder:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output folder: /content/drive/MyDrive/TESI/request_pool


## Resolve dataset versions

In [7]:
api = HfApi(token=os.environ["HF_TOKEN"])

resolved_revisions = {
    name: api.dataset_info(cfg["repo"], revision=REVISIONS[name]).sha
    for name, cfg in SOURCES.items()
}
resolved_revisions

{'routerarena': 'a4a062ce3313b56bb09c042e1bc37b61d34e3bd8',
 'magpie': '1a982eea9ece373700dd8dfd04a4de08c2578c24',
 'math': '21a5633873b6a120296cce3e2df9d5550074f4a3',
 'taco': 'c1d27565ce784104500854907dd35048c8d86b21',
 'dolly': 'bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a',
 'longbench_v2': '2b48e494f2c7a2f0af81aae178e05c7e1dde0fe9',
 'longbench_v1': '5e628be450b7e67fb7ae6e201bd6d8f7056f7672'}

## Common taxonomy

`task` keeps the variety of the sources; only true synonyms are merged. `task_group` is the coarse fallback used when capability scores for a fine task are missing.

In [8]:
TASK_GROUP = {
    "qa": "qa",
    "long_document_qa": "qa",
    "dialogue_qa": "qa",
    "reasoning": "reasoning",
    "math_problem_solving": "reasoning",
    "data_analysis": "reasoning",
    "in_context_learning": "reasoning",
    "code_generation": "code",
    "code_understanding": "code",
    "creative_writing": "generation",
    "brainstorming": "generation",
    "role_playing": "generation",
    "editing": "generation",
    "advice_seeking": "generation",
    "planning": "generation",
    "summarization": "summarization",
    "information_extraction": "extraction",
    "classification": "extraction",
    "translation": "translation",
}

DOMAINS = ["mathematics", "coding", "natural_sciences", "humanities", "social_sciences", "general"]

## Label mappings per source

In [ ]:
# RouterArena: the task is derived from the benchmark each question comes from ("Dataset name")
# rules are checked in order; any benchmark not listed is a question-answering item
ROUTERARENA_TASK_RULES = [
    ("WMT19", "translation"),
    ("NarrativeQA", "long_document_qa"),
    ("LiveCodeBench", "code_generation"),
    ("MATH", "math_problem_solving"),
    ("AIME", "math_problem_solving"),
    ("GSM8K", "math_problem_solving"),
    ("MathQA", "math_problem_solving"),
    ("AsDiv", "math_problem_solving"),
    ("MMLUPro_math", "math_problem_solving"),
    ("FinQA", "data_analysis"),
    ("Ethics_", "reasoning"),
    ("MMLU_formal_logic", "reasoning"),
    ("SuperGLUE-CausalReasoning", "reasoning"),
    ("SuperGLUE-Entailment", "reasoning"),
    ("SuperGLUE-Wsc", "reasoning"),
    ("SocialiQA", "reasoning"),
    ("ChessInstruct", "reasoning"),
]

def routerarena_task(dataset_name):
    for prefix, task in ROUTERARENA_TASK_RULES:
        if dataset_name.startswith(prefix):
            return task
    return "qa"

# RouterArena: the domain comes from the two-digit Dewey code that starts the "Category" label
ROUTERARENA_DOMAIN_BY_CODE = {
    "00": "coding",            # Computer science, knowledge, and systems
    "02": "general",           # Library and information sciences
    "10": "humanities",        # Philosophy
    "15": "social_sciences",   # Psychology
    "16": "humanities",        # Philosophical logic
    "17": "humanities",        # Ethics
    "30": "social_sciences",   # Social sciences, sociology, and anthropology
    "33": "social_sciences",   # Economics
    "34": "social_sciences",   # Law
    "36": "social_sciences",   # Social problems
    "40": "humanities",        # Language
    "50": "natural_sciences",  # Science
    "51": "mathematics",       # Mathematics
    "53": "natural_sciences",  # Physics
    "54": "natural_sciences",  # Chemistry
    "55": "natural_sciences",  # Earth sciences and geology
    "57": "natural_sciences",  # Biology
    "59": "natural_sciences",  # Animals (Zoology)
    "61": "natural_sciences",  # Medicine and health
    "62": "natural_sciences",  # Engineering
    "65": "social_sciences",   # Management and public relations
    "70": "humanities",        # Arts
    "78": "humanities",        # Music
    "79": "general",           # Sports, games and entertainment
    "80": "humanities",        # Literature, rhetoric and criticism
    "90": "humanities",        # History
    "91": "humanities",        # Geography
    "92": "humanities",        # Biography and genealogy
}

# RouterArena WMT19 items store the English sentence in "Question" and its translation in "Answer":
# the request is English -> target language, where the target is the first code of the name
# (e.g. "WMT19-de-en" -> German)
TRANSLATION_INSTRUCTION = "Translate the following text from English into {language}:"
WMT19_LANGUAGE = {
    "de": "German", "fi": "Finnish", "ru": "Russian", "cs": "Czech", "lt": "Lithuanian",
    "gu": "Gujarati", "kk": "Kazakh", "zh": "Chinese",
}

MAGPIE_TASK = {
    "Information seeking": "qa",
    "Reasoning": "reasoning",
    "Math": "math_problem_solving",
    "Coding & Debugging": "code_generation",
    "Data analysis": "data_analysis",
    "Creative writing": "creative_writing",
    "Brainstorming": "brainstorming",
    "Role playing": "role_playing",
    "Editing": "editing",
    "Advice seeking": "advice_seeking",
    "Planning": "planning",
}
MAGPIE_DOMAIN = {"Math": "mathematics", "Coding & Debugging": "coding"}   # other categories: None, classifier later

DOLLY_TASK = {
    "open_qa": "qa",
    "closed_qa": "qa",
    "general_qa": "qa",
    "classification": "classification",
    "information_extraction": "information_extraction",
    "summarization": "summarization",
    "brainstorming": "brainstorming",
    "creative_writing": "creative_writing",
}

# LongBench v2: its "domain" column holds task types, its "sub_domain" column holds subject areas
LONGBENCH_V2_TASK = {
    "Single-Document QA": "long_document_qa",
    "Multi-Document QA": "long_document_qa",
    "Long In-context Learning": "in_context_learning",
    "Code Repository Understanding": "code_understanding",
    "Long-dialogue History Understanding": "dialogue_qa",
    "Long Structured Data Understanding": "data_analysis",
}
LONGBENCH_V2_DOMAIN = {
    "Legal": "social_sciences",
    "Financial": "social_sciences",
    "Governmental": "social_sciences",
    "Literary": "humanities",
    "Detective": "humanities",
    "Code repo QA": "coding",
}   # other sub-domains (e.g. Academic, Multi-news): None, classifier later

for mapping in (MAGPIE_TASK, DOLLY_TASK, LONGBENCH_V2_TASK):
    assert set(mapping.values()) <= set(TASK_GROUP)
assert {t for _, t in ROUTERARENA_TASK_RULES} <= set(TASK_GROUP)
for mapping in (ROUTERARENA_DOMAIN_BY_CODE, MAGPIE_DOMAIN, LONGBENCH_V2_DOMAIN):
    assert set(mapping.values()) <= set(DOMAINS)

### LongBench v1 tasks and prompts

Only English tasks that fit the taxonomy are used; code completion and the synthetic tasks are excluded. The prompts are copied verbatim from `LongBench/config/dataset2prompt.json` of the THUDM/LongBench repository.

In [ ]:
LONGBENCH_V1_TASKS = {
    "hotpotqa": {
        "task": "long_document_qa",
        "prompt": "Answer the question based on the given passages. Only give me the answer and do not output any other words.\n\nThe following are given passages.\n{context}\n\nAnswer the question based on the given passages. Only give me the answer and do not output any other words.\n\nQuestion: {input}\nAnswer:",
    },
    "2wikimqa": {
        "task": "long_document_qa",
        "prompt": "Answer the question based on the given passages. Only give me the answer and do not output any other words.\n\nThe following are given passages.\n{context}\n\nAnswer the question based on the given passages. Only give me the answer and do not output any other words.\n\nQuestion: {input}\nAnswer:",
    },
    "multifieldqa_en": {
        "task": "long_document_qa",
        "prompt": "Read the following text and answer briefly.\n\n{context}\n\nNow, answer the following question based on the above text, only give me the answer and do not output any other words.\n\nQuestion: {input}\nAnswer:",
    },
    "qasper": {
        "task": "long_document_qa",
        "prompt": "You are given a scientific article and a question. Answer the question as concisely as you can, using a single phrase or sentence if possible. If the question cannot be answered based on the information in the article, write \"unanswerable\". If the question is a yes/no question, answer \"yes\", \"no\", or \"unanswerable\". Do not provide any explanation.\n\nArticle: {context}\n\n Answer the question based on the above article as concisely as you can, using a single phrase or sentence if possible. If the question cannot be answered based on the information in the article, write \"unanswerable\". If the question is a yes/no question, answer \"yes\", \"no\", or \"unanswerable\". Do not provide any explanation.\n\nQuestion: {input}\n\nAnswer:",
    },
    "gov_report": {
        "task": "summarization",
        "prompt": "You are given a report by a government agency. Write a one-page summary of the report.\n\nReport:\n{context}\n\nNow, write a one-page summary of the report.\n\nSummary:",
    },
    "multi_news": {
        "task": "summarization",
        "prompt": "You are given several news passages. Write a one-page summary of all news. \n\nNews:\n{context}\n\nNow, write a one-page summary of all the news.\n\nSummary:",
    },
    "samsum": {
        "task": "summarization",
        "prompt": "Summarize the dialogue into a few short sentences. The following are some examples.\n\n{context}\n\n{input}",
    },
    "triviaqa": {
        "task": "in_context_learning",
        "prompt": "Answer the question based on the given passage. Only give me the answer and do not output any other words. The following are some examples.\n\n{context}\n\n{input}",
    },
    "trec": {
        "task": "in_context_learning",
        "prompt": "Please determine the type of the question below. Here are some examples of questions.\n\n{context}\n{input}",
    },
}
assert {cfg["task"] for cfg in LONGBENCH_V1_TASKS.values()} <= set(TASK_GROUP)

def longbench_v1_length_bin(n_words):
    # the three LongBench-E length ranges in words
    if n_words < 4000:
        return "0-4k_words"
    if n_words < 8000:
        return "4-8k_words"
    return "8k+_words"

## Helpers

In [ ]:
CJK_RE = re.compile(r"[぀-ヿ㐀-䶿一-鿿가-힯]")

def cjk_ratio(text, sample_chars=20_000):
    sample = text[:sample_chars]
    return len(CJK_RE.findall(sample)) / max(len(sample), 1)

def join_parts(*parts):
    # joins the non-empty pieces of a request with a blank line
    return "\n\n".join(p.strip() for p in parts if isinstance(p, str) and p.strip())

def format_options(options, letters="ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    if options is None or len(options) == 0:
        return ""
    return "\n".join(f"{letters[i]}. {option}" for i, option in enumerate(options))

def parse_listlike(value):
    # TACO stores some lists as their string representation
    if isinstance(value, (list, tuple, np.ndarray)):
        return list(value)
    if isinstance(value, str) and value.strip().startswith("["):
        try:
            return list(ast.literal_eval(value))
        except (ValueError, SyntaxError):
            return []
    return []

def first_solution(value):
    try:
        solutions = json.loads(value) if isinstance(value, str) else list(value)
    except (ValueError, TypeError):
        return None
    return solutions[0] if len(solutions) > 0 and isinstance(solutions[0], str) else None

def normalize_for_dedup(text):
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()

CELL_COLUMNS = ["task", "domain", "sampling_stratum"]

def cell_sample(df, n=CELL_SIZE, seed=SEED):
    # same number of requests from every (task, domain, sampling_stratum) cell of one source
    # a cell with fewer requests is taken whole
    keys = df[CELL_COLUMNS].astype(object).where(df[CELL_COLUMNS].notna(), "<none>")
    return pd.concat(
        group.sample(n=min(len(group), n), random_state=seed)
        for _, group in df.groupby([keys[c] for c in CELL_COLUMNS], sort=True)
    )

def log_counts(source, available, sampled):
    BUILD_LOG[source] = {
        "available_after_source_filters": int(len(available)),
        "n_cells": int(available[CELL_COLUMNS].astype(object).fillna("<none>").drop_duplicates().shape[0]),
        "sampled": int(len(sampled)),
    }

SCHEMA = [
    "source", "source_dataset", "source_index", "text",
    "task_raw", "task", "domain_raw", "domain", "domain_fine",
    "difficulty_raw", "difficulty_scale", "sampling_stratum", "reference_output",
]

## Source adapters

Each adapter loads one source, maps it to the common schema, applies the source-specific filters and draws its cell sample. LongBench v1 has no difficulty, so its length range is used as the sampling stratum.

In [12]:
def load_routerarena(cfg, revision):
    df = load_dataset(cfg["repo"], split=cfg["split"], revision=revision).to_pandas()
    is_translation = df["Dataset name"].str.startswith("WMT19")
    target_language = df["Dataset name"].str.split("-").str[1].map(WMT19_LANGUAGE)
    assert target_language[is_translation].notna().all(), "unknown WMT19 language code"

    def request_text(question, context, options, translation, language):
        if translation:
            return join_parts(TRANSLATION_INSTRUCTION.format(language=language), question)
        return join_parts(context, question, format_options(options))   # what the model reads

    out = pd.DataFrame({
        "source": "routerarena",
        "source_dataset": df["Dataset name"],
        "source_index": df["Global Index"].astype(str),
        "text": [request_text(*args) for args in zip(df["Question"], df["Context"], df["Options"],
                                                     is_translation, target_language)],
        "task_raw": df["Dataset name"],
        "task": df["Dataset name"].map(routerarena_task),
        "domain_raw": df["Domain"],
        "domain": df["Category"].str[:2].map(ROUTERARENA_DOMAIN_BY_CODE),
        "domain_fine": df["Category"],
        "difficulty_raw": df["Difficulty"],
        "difficulty_scale": "routerarena_easy_medium_hard",
        "sampling_stratum": df["Difficulty"],
        # Other answers are mostly an option letter, not a realistic output length.
        "reference_output": df["Answer"].where(is_translation, None),
    })
    out = out[out["text"] != ""]
    sampled = cell_sample(out)
    log_counts("routerarena", out, sampled)
    return sampled


def load_magpie(cfg, revision):
    ds = load_dataset(cfg["repo"], split=cfg["split"], revision=revision)
    meta = ds.select_columns(["task_category", "difficulty", "language"]).to_pandas()
    meta["row"] = np.arange(len(meta))
    meta = meta[(meta["language"] == "EN")
                & meta["task_category"].isin(list(MAGPIE_TASK))
                & meta["difficulty"].notna()].copy()
    meta["task"] = meta["task_category"].map(MAGPIE_TASK)
    meta["domain"] = meta["task_category"].map(MAGPIE_DOMAIN)
    meta["sampling_stratum"] = meta["difficulty"]
    sampled = cell_sample(meta).reset_index(drop=True)
    log_counts("magpie", meta, sampled)
    rows = ds.select(sampled["row"].tolist()).select_columns(["uuid", "instruction", "response"]).to_pandas()
    return pd.DataFrame({
        "source": "magpie",
        "source_dataset": "magpie",
        "source_index": rows["uuid"],
        "text": rows["instruction"].str.strip(),
        "task_raw": sampled["task_category"],
        "task": sampled["task"],
        "domain_raw": sampled["task_category"],
        "domain": sampled["domain"],
        "domain_fine": None,
        "difficulty_raw": sampled["difficulty"],
        "difficulty_scale": "magpie_very_easy_to_very_hard",
        "sampling_stratum": sampled["sampling_stratum"],
        "reference_output": rows["response"],
    })


MATH_SUBJECTS = ["algebra", "counting_and_probability", "geometry", "intermediate_algebra",
                 "number_theory", "prealgebra", "precalculus"]

def load_math(cfg, revision):
    parts = []
    for subject in MATH_SUBJECTS:
        for split in cfg["split"]:
            d = load_dataset(cfg["repo"], subject, split=split, revision=revision).to_pandas()
            d["source_index"] = [f"{subject}/{split}/{i}" for i in range(len(d))]
            parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df = df[df["level"].str.fullmatch(r"Level [1-5]")]      # drops the few "Level ?" problems
    out = pd.DataFrame({
        "source": "math",
        "source_dataset": "math",
        "source_index": df["source_index"],
        "text": df["problem"].str.strip(),
        "task_raw": "competition_math",
        "task": "math_problem_solving",
        "domain_raw": df["type"],
        "domain": "mathematics",
        "domain_fine": df["type"],
        "difficulty_raw": df["level"],
        "difficulty_scale": "math_level_1_to_5",
        "sampling_stratum": df["level"],
        "reference_output": df["solution"],
    })
    sampled = cell_sample(out)
    log_counts("math", out, sampled)
    return sampled


def load_taco(cfg, revision):
    ds = load_dataset(cfg["repo"], split=cfg["split"], revision=revision)
    meta = ds.select_columns(["difficulty", "skill_types"]).to_pandas()
    meta["row"] = np.arange(len(meta))
    meta = meta[meta["difficulty"] != "UNKNOWN_DIFFICULTY"].copy()
    meta["task"], meta["domain"], meta["sampling_stratum"] = "code_generation", "coding", meta["difficulty"]
    sampled = cell_sample(meta).reset_index(drop=True)
    log_counts("taco", meta, sampled)
    rows = ds.select(sampled["row"].tolist()).select_columns(
        ["question", "starter_code", "solutions", "url"]).to_pandas()
    starter = rows["starter_code"].fillna("").map(lambda s: f"Starter code:\n{s}" if s.strip() else "")
    skills = sampled["skill_types"].map(parse_listlike)
    return pd.DataFrame({
        "source": "taco",
        "source_dataset": "taco",
        "source_index": rows["url"].fillna("").astype(str) + "#" + sampled["row"].astype(str),
        "text": [join_parts(q, s) for q, s in zip(rows["question"], starter)],
        "task_raw": "competitive_programming",
        "task": "code_generation",
        "domain_raw": "code",
        "domain": "coding",
        "domain_fine": skills.map(lambda s: "; ".join(s) if s else None),
        "difficulty_raw": sampled["difficulty"],
        "difficulty_scale": "taco_easy_to_very_hard",
        "sampling_stratum": sampled["sampling_stratum"],
        "reference_output": rows["solutions"].map(first_solution),
    })


def load_dolly(cfg, revision):
    df = load_dataset(cfg["repo"], split=cfg["split"], revision=revision).to_pandas()
    out = pd.DataFrame({
        "source": "dolly",
        "source_dataset": "dolly",
        "source_index": np.arange(len(df)).astype(str),
        "text": [join_parts(i, c) for i, c in zip(df["instruction"], df["context"])],
        "task_raw": df["category"],
        "task": df["category"].map(DOLLY_TASK),
        "domain_raw": None,
        "domain": None,          # Dolly has no domain label: classifier later
        "domain_fine": None,
        "difficulty_raw": None,
        "difficulty_scale": "none",
        "sampling_stratum": None,
        "reference_output": df["response"],
    })
    sampled = cell_sample(out)
    log_counts("dolly", out, sampled)
    return sampled


def load_longbench_v2(cfg, revision):
    df = load_dataset(cfg["repo"], split=cfg["split"], revision=revision).to_pandas()
    options = df[["choice_A", "choice_B", "choice_C", "choice_D"]].values.tolist()
    df["text"] = [join_parts(c, f"Question: {q}", format_options(o))
                  for c, q, o in zip(df["context"], df["question"], options)]
    is_chinese = df.apply(lambda r: cjk_ratio(r["question"] + "\n" + r["context"]) > 0.05, axis=1)
    too_long = df["text"].str.len() > MAX_INPUT_TOKENS * CHARS_PER_TOKEN_UPPER
    BUILD_LOG["longbench_v2_filters"] = {
        "loaded": int(len(df)),
        "removed_chinese": int(is_chinese.sum()),
        "removed_certainly_over_token_limit": int((too_long & ~is_chinese).sum()),
    }
    df = df[~is_chinese & ~too_long]
    out = pd.DataFrame({
        "source": "longbench_v2",
        "source_dataset": "longbench_v2",
        "source_index": df["_id"].astype(str),
        "text": df["text"],
        "task_raw": df["domain"],
        "task": df["domain"].map(LONGBENCH_V2_TASK),
        "domain_raw": df["sub_domain"],
        "domain": df["sub_domain"].map(LONGBENCH_V2_DOMAIN),
        "domain_fine": df["sub_domain"],
        "difficulty_raw": df["difficulty"],
        "difficulty_scale": "longbench_easy_hard",
        "sampling_stratum": df["difficulty"],
        "reference_output": None,   # the answer is an option letter
    })
    sampled = cell_sample(out)
    log_counts("longbench_v2", out, sampled)
    return sampled


def load_longbench_v1(cfg, revision):
    zip_path = hf_hub_download(cfg["repo"], cfg["file"], repo_type="dataset",
                               revision=revision, token=os.environ["HF_TOKEN"])
    rows = []
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.namelist()
        for name, task_cfg in LONGBENCH_V1_TASKS.items():
            filename = f"{name}_e.jsonl"
            member = next(m for m in members if m == filename or m.endswith("/" + filename))
            with archive.open(member) as f:
                for i, line in enumerate(f):
                    r = json.loads(line)
                    rows.append({
                        "name": name, "i": i, "id": r.get("_id", f"{name}_{i}"),
                        "language": r.get("language", "en"), "length": r["length"],
                        "text": task_cfg["prompt"].format(context=r["context"], input=r["input"]).strip(),
                        "answer": (r.get("answers") or [None])[0],
                    })
    df = pd.DataFrame(rows)
    df = df[(df["language"] == "en")
            & (df["text"].str.len() <= MAX_INPUT_TOKENS * CHARS_PER_TOKEN_UPPER)]
    task = df["name"].map(lambda n: LONGBENCH_V1_TASKS[n]["task"])
    out = pd.DataFrame({
        "source": "longbench_v1",
        "source_dataset": df["name"],
        "source_index": df["id"].astype(str),
        "text": df["text"],
        "task_raw": df["name"],
        "task": task,
        "domain_raw": None,
        "domain": None,          # no domain label: classifier later
        "domain_fine": None,
        "difficulty_raw": None,
        "difficulty_scale": "none",
        "sampling_stratum": df["length"].map(longbench_v1_length_bin),
        # Summaries are realistic outputs; QA and few-shot answers are short gold spans.
        "reference_output": df["answer"].where(task == "summarization", None),
    })
    sampled = cell_sample(out)
    log_counts("longbench_v1", out, sampled)
    return sampled


LOADERS = {
    "routerarena": load_routerarena,
    "magpie": load_magpie,
    "math": load_math,
    "taco": load_taco,
    "dolly": load_dolly,
    "longbench_v2": load_longbench_v2,
    "longbench_v1": load_longbench_v1,
}
assert set(LOADERS) == set(SOURCES)

## Load and convert all sources

In [13]:
frames = {}
for name, cfg in SOURCES.items():
    frame = LOADERS[name](cfg, REVISIONS[name])[SCHEMA]
    assert frame["task"].notna().all(), f"{name}: some original task labels have no mapping"
    frames[name] = frame

pd.DataFrame({k: v for k, v in BUILD_LOG.items() if k in SOURCES}).T

README.md:   0%|          | 0.00/2.64k [00:00<?, ?B/s]

data/sub_10-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  380kB            

data/sub_10-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/full-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.67MB            

data/full-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/robustness-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  233kB            

... (69 outputs left out)

Generating train split:   0%|          | 0/503 [00:00<?, ? examples/s]

data.zip: reconstructing file:   0%|          |  0.00B /  114MB            

data.zip: downloading bytes:           |  0.00B            

,available_after_source_filters,n_cells,sampled
routerarena,8400,39,1587
magpie,296753,53,1932
math,12498,5,250
taco,7070,5,250
dolly,15011,6,300
longbench_v2,377,14,334
longbench_v1,2468,9,450


## Merge and remove duplicates

Duplicates are found on the normalized text (Unicode-normalized, lower case, single spaces). When the same request appears in two sources, the copy from the first source in `DEDUP_PRIORITY` is kept: specialized sources come first because their labels are richer.

In [14]:
DEDUP_PRIORITY = ["math", "taco", "longbench_v2", "longbench_v1", "dolly", "magpie", "routerarena"]
assert set(DEDUP_PRIORITY) == set(SOURCES)

pool = pd.concat([frames[name] for name in DEDUP_PRIORITY], ignore_index=True)

# One representation for missing labels (sources use both None and NaN).
LABEL_COLUMNS = ["task_raw", "domain_raw", "domain", "domain_fine", "difficulty_raw", "sampling_stratum"]
pool[LABEL_COLUMNS] = pool[LABEL_COLUMNS].astype(object).where(pool[LABEL_COLUMNS].notna(), None)

pool["dedup_key"] = pool["text"].map(normalize_for_dedup)
duplicate = pool.duplicated("dedup_key", keep="first")
removed_duplicates = pool.loc[duplicate, "source"].value_counts()
for name in SOURCES:
    BUILD_LOG[name]["removed_duplicates"] = int(removed_duplicates.get(name, 0))

pool = pool[~duplicate].drop(columns="dedup_key").reset_index(drop=True)
print(f"Duplicates removed: {int(duplicate.sum())}. Pool after deduplication: {len(pool):,} requests")

Duplicates removed: 0. Pool after deduplication: 5,103 requests


## Token counts

`input_tokens` is measured on the text the model reads. `reference_output_tokens` is measured on the reference answer where it is a realistic answer (Magpie, MATH, TACO, Dolly, RouterArena translations, LongBench v1 summaries), and is empty otherwise.

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, token=os.environ["HF_TOKEN"])
tokenizer.model_max_length = 10**9   # counting only: silences the long-sequence warning

def count_tokens(texts, batch_size=256):
    texts = list(texts)
    counts = np.full(len(texts), np.nan)
    present = [i for i, t in enumerate(texts) if isinstance(t, str) and t]
    for start in range(0, len(present), batch_size):
        idx = present[start:start + batch_size]
        encoded = tokenizer([texts[i] for i in idx], add_special_tokens=False)["input_ids"]
        counts[idx] = [len(e) for e in encoded]
    return counts

pool["input_tokens"] = count_tokens(pool["text"]).astype(int)
pool["reference_output_tokens"] = count_tokens(pool["reference_output"])
pool = pool.drop(columns="reference_output")

over_limit = pool["input_tokens"] > MAX_INPUT_TOKENS
removed_over_limit = pool.loc[over_limit, "source"].value_counts()
for name in SOURCES:
    BUILD_LOG[name]["removed_over_token_limit"] = int(removed_over_limit.get(name, 0))
print(f"Removed {int(over_limit.sum())} requests over {MAX_INPUT_TOKENS:,} tokens")
pool = pool[~over_limit].reset_index(drop=True)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Removed 61 requests over 128,000 tokens


## Final columns

In [16]:
pool["task_group"] = pool["task"].map(TASK_GROUP)
pool["request_id"] = pool["source"] + "-" + pool.groupby("source").cumcount().map("{:05d}".format)

COLUMNS = [
    "request_id", "source", "source_dataset", "source_index", "text",
    "task", "task_group", "task_raw",
    "domain", "domain_fine", "domain_raw",
    "difficulty_raw", "difficulty_scale", "sampling_stratum",
    "input_tokens", "reference_output_tokens",
]
pool = pool[COLUMNS]

final_counts = pool["source"].value_counts()
for name in SOURCES:
    BUILD_LOG[name]["final"] = int(final_counts.get(name, 0))

build_log = pd.DataFrame({k: v for k, v in BUILD_LOG.items() if k in SOURCES}).T
build_log.loc["total"] = build_log.sum()
build_log

,available_after_source_filters,n_cells,sampled,removed_duplicates,removed_over_token_limit,final
routerarena,8400,39,1587,0,0,1587
magpie,296753,53,1932,0,0,1932
math,12498,5,250,0,0,250
taco,7070,5,250,0,0,250
dolly,15011,6,300,0,0,300
longbench_v2,377,14,334,0,61,273
longbench_v1,2468,9,450,0,0,450
total,342577,131,5103,0,61,5042


## Checks

In [17]:
assert pool["request_id"].is_unique
assert pool["text"].str.len().gt(0).all()
assert pool["task"].isin(list(TASK_GROUP)).all()
assert pool["task_group"].notna().all()
assert pool["domain"].dropna().isin(DOMAINS).all()
assert pool["input_tokens"].between(1, MAX_INPUT_TOKENS).all()
assert set(pool["source"]) == set(SOURCES), "a source lost all its requests"
translation = pool[pool["task"] == "translation"]
assert translation["text"].str.startswith("Translate the following text").all()
print("All checks passed.")

All checks passed.


## Save

In [18]:
pool_path = os.path.join(OUTPUT_DIR, f"request_pool_{POOL_VERSION}.parquet")
pool.to_parquet(pool_path, index=False)

# The label mappings actually applied, for documentation in the thesis.
task_mapping = (pool.groupby(["source", "task_raw", "task", "task_group"]).size()
                    .rename("n_requests").reset_index())
domain_mapping = (pool.groupby(["source", "domain_raw", "domain"], dropna=False).size()
                      .rename("n_requests").reset_index())
task_mapping.to_csv(os.path.join(OUTPUT_DIR, f"task_mapping_{POOL_VERSION}.csv"), index=False)
domain_mapping.to_csv(os.path.join(OUTPUT_DIR, f"domain_mapping_{POOL_VERSION}.csv"), index=False)

metadata = {
    "pool_version": POOL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED,
    "cell_size": CELL_SIZE,
    "cell_columns": CELL_COLUMNS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "tokenizer": TOKENIZER_ID,
    "translation_instruction": TRANSLATION_INSTRUCTION,
    "longbench_v1_tasks": {name: c["task"] for name, c in LONGBENCH_V1_TASKS.items()},
    "longbench_v1_prompts_source": "https://github.com/THUDM/LongBench/blob/main/LongBench/config/dataset2prompt.json",
    "sources": {name: {**cfg, "revision": resolved_revisions[name]} for name, cfg in SOURCES.items()},
    "dedup_priority": DEDUP_PRIORITY,
    "build_log": BUILD_LOG,
    "n_requests": int(len(pool)),
}
with open(os.path.join(OUTPUT_DIR, f"request_pool_{POOL_VERSION}_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", pool_path)

Saved: /content/drive/MyDrive/TESI/request_pool/request_pool_v2.parquet
